In [0]:
import requests
import json
from pyspark.sql import Row

dbutils.widgets.text("tmdb_api_key", "")
API_KEY = dbutils.widgets.get("tmdb_api_key")
BASE_URL = "https://api.themoviedb.org/3"

In [0]:
def fetch_languages():
    resp = requests.get(f"{BASE_URL}/configuration/languages", params={"api_key": API_KEY})
    resp.raise_for_status()
    return resp.json() 

languages = fetch_languages()
languages_df = spark.createDataFrame([Row(**l) for l in languages])
languages_df.write.format("delta").mode("overwrite").saveAsTable("moviebuff.default.bronze_languages")

print(f"Ingested {len(languages)} languages.")
languages_df.show(5)

Get data

In [0]:
def fetch_genres():
    resp = requests.get(f"{BASE_URL}/genre/movie/list", params={"api_key": API_KEY})
    resp.raise_for_status()
    return resp.json()["genres"]

genres = fetch_genres()
genres_df = spark.createDataFrame([Row(**g) for g in genres])

# saveAsTable lets Databricks pick the storage location — no DBFS root path needed
genres_df.write.format("delta").mode("overwrite").saveAsTable("moviebuff.default.bronze_genres")

print(f"Ingested {len(genres)} genres.")


In [0]:
spark.sql("SELECT * FROM moviebuff.default.bronze_genres LIMIT 5").show()

In [0]:
import time

def fetch_year(year, min_vote_count=50, pages_per_year=500):
    movies = []
    for page in range(1, pages_per_year + 1):
        resp = requests.get(
            f"{BASE_URL}/discover/movie",
            params={
                "api_key": API_KEY,
                "sort_by": "popularity.desc",
                "page": page,
                "primary_release_year": year,
                "vote_count.gte": min_vote_count
            }
        )
        resp.raise_for_status()
        results = resp.json().get("results", [])
        if not results:
            break
        movies.extend(results)
        time.sleep(0.02)
    return movies

def fetch_and_save_by_year(start_year=1970, end_year=2026, min_vote_count=50):
    total_saved = 0
    for year in range(start_year, end_year + 1):
        movies = fetch_year(year, min_vote_count=min_vote_count)
        
        if not movies:
            print(f"Year {year}: 0 movies, skipping")
            continue
        
        year_df = spark.createDataFrame(
            [Row(raw_json=json.dumps(m)) for m in movies]
        )
        
        # append after EVERY year — this is the checkpoint
        year_df.write.format("delta").mode("append").saveAsTable("moviebuff.default.bronze_movies_raw")
        
        total_saved += len(movies)
        print(f"Year {year}: saved {len(movies)} movies. Running total: {total_saved}")
    
    print(f"DONE. Total movies saved across all years: {total_saved}")

# first, clear the table once so we start clean (only run this ONE time)
spark.sql("DROP TABLE IF EXISTS moviebuff.default.bronze_movies_raw")

# now run the checkpointed fetch
fetch_and_save_by_year(start_year=1970, end_year=2026, min_vote_count=50)

In [0]:
%sql
SELECT * FROM moviebuff.default.bronze_movies_raw 

In [0]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, IntegerType

id_schema = StructType([StructField("id", IntegerType())])

movies_table = spark.table("moviebuff.default.bronze_movies_raw")

all_movie_ids = [
    row["id"] for row in
    movies_table.withColumn("data", from_json(col("raw_json"), id_schema))
                .select("data.id")
                .distinct()
                .collect()
]

print(f"Total movie IDs loaded from table: {len(all_movie_ids)}")

This notebook fetch the the data into bronze table through api request

In [0]:
import time

TARGET_COUNTRIES = ["IN", "IE"]
BATCH_SIZE = 500  # write to Delta every 500 movies processed

def fetch_watch_providers(movie_id):
    resp = requests.get(
        f"{BASE_URL}/movie/{movie_id}/watch/providers",
        params={"api_key": API_KEY}
    )
    resp.raise_for_status()
    return resp.json().get("results", {})



# resume support: skip movie_ids already fetched
try:
    already_done = set(
        row["movie_id"] for row in spark.sql(
            "SELECT DISTINCT movie_id FROM moviebuff.default.bronze_watch_providers_raw"
        ).collect()
    )
    print(f"Already fetched providers for {len(already_done)} movies, will skip those")
except Exception:
    already_done = set()
    print("No existing provider table found, starting fresh")

movie_ids_to_fetch = [mid for mid in all_movie_ids if mid not in already_done]
print(f"Movies remaining to fetch: {len(movie_ids_to_fetch)}")

provider_records = []
total_saved = 0

for i, mid in enumerate(movie_ids_to_fetch):
    try:
        data = fetch_watch_providers(mid)
        for country in TARGET_COUNTRIES:
            country_data = data.get(country)
            if country_data:
                provider_records.append(
                    Row(
                        movie_id=mid,
                        country=country,
                        raw_json=json.dumps(country_data)
                    )
                )
    except Exception as e:
        print(f"Failed for movie_id {mid}: {e}")
    
    time.sleep(0.02)
    
    # checkpoint every BATCH_SIZE movies
    if (i + 1) % BATCH_SIZE == 0 or (i + 1) == len(movie_ids_to_fetch):
        if provider_records:
            batch_df = spark.createDataFrame(provider_records)
            batch_df.write.format("delta").mode("append").saveAsTable("moviebuff.default.bronze_watch_providers_raw")
            total_saved += len(provider_records)
            print(f"Checkpoint: processed {i+1}/{len(movie_ids_to_fetch)} movies, saved {len(provider_records)} records (total saved: {total_saved})")
            provider_records = []  # clear buffer after writing

print(f"DONE. Total new provider records saved: {total_saved}")

In [0]:
%sql
SELECT count(*) FROM moviebuff.default.bronze_watch_providers_raw LIMIT 10;

In [0]:
##to check count of years

existing_df = spark.table("bronze_movies_raw")

from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# minimal schema just to extract id and release_date
check_schema = StructType([
    StructField("id", IntegerType()),
    StructField("release_date", StringType())
])

existing_parsed = existing_df.withColumn(
    "data", from_json(col("raw_json"), check_schema)
).select("data.id", "data.release_date")

existing_parsed.createOrReplaceTempView("existing_movies")

# distinct years already covered
spark.sql("""
    SELECT SUBSTRING(release_date, 1, 4) as year, COUNT(*) as count
    FROM existing_movies
    WHERE release_date IS NOT NULL AND release_date != ''
    GROUP BY year
    ORDER BY year
""").show(100)

In [0]:
existing_ids = set(
    row["id"] for row in spark.sql("SELECT DISTINCT id FROM existing_movies").collect()
)
print(f"Already have {len(existing_ids)} unique movies")